# Compare OLD vs NEW `agent_response` extraction

Sanity-checks the strict `agent_answer`-span extractor against the old
"longest stopped AI message" heuristic on the same raw MLflow traces.

What this does:
1. Loads the cached `traces_{RUN_NAME}.pickle` written by `czkb_001_import_traces_local.ipynb`.
2. Re-derives the OLD `agent_response` using a self-contained replica of the
   deleted `extract_final_ai_content` heuristic.
3. Computes the NEW `agent_response` using `extract_agent_answer_span`.
4. Diffs row-by-row and surfaces the categories (same, both empty,
   old‑only, new‑only, both‑but‑different).

Rows where OLD is non-empty and NEW differs (or is empty) are the silent
errors the heuristic used to produce — those are the trace IDs that any
downstream LLM-judge / scorer was reading the wrong answer for.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

import pandas as pd

# Make the hg-ds-evals repo importable when running off-cluster.
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / "hg_ds_evals").is_dir():
    _repo_root = _repo_root.parent
if (_repo_root / "hg_ds_evals").is_dir() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
    print(f'repo root added to sys.path: "{_repo_root}"')

from hg_ds_evals.preprocessing.traces import (
    AI_TYPE_STRINGS,
    extract_agent_answer_span,
    normalize_mlflow_trace_row,
)
from hg_ds_evals.preprocessing.traces import _parse_attr_json, _walk_dicts  # internal helpers

## Run selection

Use the same key you used in `czkb_001_import_traces_local.ipynb`. The
pickle path is derived the same way (`traces_{RUN_NAME}.pickle` under
`../input/`).

In [9]:
all_runs: dict = {
    "czkb_jorao": {
        "experiment_id": "471458066277040",
        "run_id": "11831eebd37a445a889d124baec45a32",
        "mlflow_run_name": "online/adhoc/quiet-hawk/score",
    },
    "april_30": {
        "experiment_id": "471458066277040",
        "run_id": "76ae7ceaf4e94423a838ab70c5493c5a",
        "mlflow_run_name": "online/nightly/2026-04-30/score",
    },
    "may_1": {
        "experiment_id": "471458066277040",
        "run_id": "db7ae47eee0348f8a3fd9c6dd21b5b3a",
        "mlflow_run_name": "online/nightly/2026-05-01/score",
    },
    "april_24": {
        "experiment_id": "471458066277040",
        "run_id": "3441808bc8db416fbbce3f4878e94d4d",
        "mlflow_run_name": "online_nightly_2026_04_24_infer",
    },
}

RUN_DATE = "april_30"
RUN_NAME = all_runs[RUN_DATE]["mlflow_run_name"].replace("/", "_").replace("-", "_")
traces_pickle_path = Path(f"../input/traces_{RUN_NAME}.pickle")
print(f"RUN_NAME:           {RUN_NAME}")
print(f"traces_pickle_path: {traces_pickle_path}")
print(f"exists:             {traces_pickle_path.is_file()}")

RUN_NAME:           online_nightly_2026_04_30_score
traces_pickle_path: ../input/traces_online_nightly_2026_04_30_score.pickle
exists:             True


## Load raw MLflow traces

In [10]:
if not traces_pickle_path.is_file():
    raise FileNotFoundError(
        f"No cached traces at {traces_pickle_path}. Run czkb_001_import_traces_local.ipynb "
        "first (FORCE_REFETCH_TRACES=True) to populate the pickle."
    )

traces_df = pd.read_pickle(traces_pickle_path)
print(f"traces loaded: {len(traces_df)}")
print(f"columns:       {list(traces_df.columns)}")

traces loaded: 1123
columns:       ['trace_id', 'trace', 'client_request_id', 'state', 'request_time', 'execution_duration', 'request', 'response', 'trace_metadata', 'tags', 'spans', 'assessments']


## Define the OLD heuristic extractor (inline replica)

This is the deleted `extract_final_ai_content` from the previous
`skkb_traces.py`, reproduced here verbatim so we can compare apples to
apples without depending on a stale checkpoint. Rule: scan every span's
`mlflow.spanOutputs`, find AI-typed nodes where `finish_reason` is
`"stop"` or absent, return the longest `content` string.

In [5]:
def extract_final_ai_content_old(spans):
    """Replica of the deleted heuristic. Do not use elsewhere."""
    best = ""
    for span in spans:
        parsed = _parse_attr_json(span.get("attributes") or {}, "mlflow.spanOutputs")
        if parsed is None:
            continue
        for node in _walk_dicts(parsed):
            node_type = node.get("type")
            if node_type not in AI_TYPE_STRINGS and str(node_type).lower() != "ai":
                continue
            content = node.get("content")
            if not isinstance(content, str) or not content.strip():
                continue
            response_metadata = node.get("response_metadata") or {}
            finish_reason = response_metadata.get("finish_reason")
            if finish_reason is not None and finish_reason != "stop":
                continue
            if len(content) > len(best):
                best = content
    return best

## Run both extractors over every trace

In [11]:
rows = []
for raw_row in traces_df.to_dict(orient="records"):
    canonical = normalize_mlflow_trace_row(raw_row)
    spans = canonical["data"]["spans"]
    old_answer = extract_final_ai_content_old(spans)
    new_answer = extract_agent_answer_span(spans)
    rows.append({
        "trace_id": canonical["info"].get("trace_id"),
        "agent_response_old": old_answer,
        "agent_response_new": new_answer,
        "old_len": len(old_answer),
        "new_len": len(new_answer),
        "same": old_answer == new_answer,
    })

cmp_df = pd.DataFrame(rows)
print(f"rows compared: {len(cmp_df)}")
cmp_df.head()

rows compared: 1123


,trace_id,agent_response_old,agent_response_new,old_len,new_len,same
0,tr-da7870414968109e375f3818e98249b9,"Ano, můžete. Účet v USD si jako dospělá klient...",,461,0,False
1,tr-5a022cc33b801464f3c88f6fd8c2bae1,Do vkladového bankomatu České spořitelny bohuž...,,239,0,False
2,tr-910e6c9636eb107a30253f5638d7ce21,"Ano, na Plus účet vedený v Kč můžete přijímat ...",,380,0,False
3,tr-ce6ab122069328f34cafec3b940d4348,Eurový účet je u nás „Účet v cizí měně“ v EUR....,,559,0,False
4,tr-fd24236d4ed27556a796e74606044bcc,"Eura můžete vložit dvěma způsoby, ale vždy pře...",,943,0,False


## Summary categories

- **both_empty**            — no canonical span and no AI content anywhere; both extractors return `""`
- **same_nonempty**         — heuristic happened to land on the canonical answer
- **old_only**              — OLD produced text, NEW is empty → the heuristic was guessing on traces with no `agent_answer` span
- **new_only**              — NEW produced text, OLD is empty → canonical span exists but the heuristic's `finish_reason` filter rejected it
- **different_both_nonempty** — both produced text but they disagree → the heuristic was picking a longer non-final AI message (silent error)

In [12]:
def categorize(row):
    old_has = bool(row["agent_response_old"])
    new_has = bool(row["agent_response_new"])
    if not old_has and not new_has:
        return "both_empty"
    if old_has and new_has and row["same"]:
        return "same_nonempty"
    if old_has and not new_has:
        return "old_only"
    if new_has and not old_has:
        return "new_only"
    return "different_both_nonempty"

cmp_df["category"] = cmp_df.apply(categorize, axis=1)
counts = cmp_df["category"].value_counts(dropna=False)
print(counts.to_string())
print(f"\nrows where the old heuristic was wrong (old_only + different_both_nonempty): "
      f"{int((cmp_df['category'].isin(['old_only', 'different_both_nonempty'])).sum())} "
      f"/ {len(cmp_df)}")
print(f"rows the heuristic missed but the new extractor recovers (new_only): "
      f"{int((cmp_df['category'] == 'new_only').sum())} / {len(cmp_df)}")

category
old_only      1122
both_empty       1

rows where the old heuristic was wrong (old_only + different_both_nonempty): 1122 / 1123
rows the heuristic missed but the new extractor recovers (new_only): 0 / 1123


## Example differences

Inspect a handful from each non-trivial category to eyeball what changed.
Truncates long strings for readability.

In [8]:
def show_examples(df, category, n=3, width=240):
    sub = df[df["category"] == category].head(n)
    if sub.empty:
        print(f"=== {category}: 0 rows ===\n")
        return
    print(f"=== {category}: {len(df[df['category'] == category])} rows (showing {len(sub)}) ===\n")
    for _, row in sub.iterrows():
        print(f"trace_id: {row['trace_id']}")
        print(f"  old ({row['old_len']:>5} ch): {row['agent_response_old'][:width]!r}")
        print(f"  new ({row['new_len']:>5} ch): {row['agent_response_new'][:width]!r}")
        print()

for category in ("different_both_nonempty", "old_only", "new_only"):
    show_examples(cmp_df, category, n=5)

=== different_both_nonempty: 0 rows ===

=== old_only: 0 rows ===

=== new_only: 0 rows ===



## Optional: persist for offline inspection

In [ ]:
out_path = Path(f"traces/agent_response_diff_{RUN_NAME}.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
cmp_df.to_csv(out_path, index=False)
print(f"saved -> {out_path}")